# SGLang Bridge — Integration Test

End-to-end validation of `boot_sglang` on a real GPU. Runs the v4 Driver protocol's SGLang backend against `meta-llama/Llama-3.2-1B`, compares per-hook activations and next-token argmax against the HF transformers backend, exercises the affine intervention path, and verifies tensor read-back over the ZMQ side channel.

Sibling notebook to `vLLM_Bridge_Integration_Test.ipynb`. Same acceptance shape; different inference engine and a fundamentally different worker integration:

## Architecture (v4 corrected design)

SGLang spawns its `Scheduler` in a separate subprocess via `multiprocessing.spawn`. There's no in-process mode. Our integration therefore avoids any patch-in-the-driver-then-construct-Engine pattern (the patch never crosses spawn). Instead:

1. **Install side** — `ServerArgs.forward_hooks` carries JSON specs from driver to worker. Each spec names a factory dotted path (`transformer_lens.model_bridge.sources.sglang.hooks:make_capture_hook`); SGLang `importlib`-resolves it inside the worker, after `load_model()` and before `init_piecewise_cuda_graphs()`, and registers the returned callable on every module matching the spec's `target_modules` fnmatch pattern.
2. **Plugin** — a setuptools `[project.entry-points."sglang.srt.plugins"]` entry on our package. SGLang's `load_plugins()` (first line of `run_scheduler_process` in the worker, first line of `Engine.__init__` in the driver) invokes our `worker_plugin.register()` in each process, which `setattr`s our `tl_*` methods onto the `Scheduler` class. After this, `engine.collective_rpc("tl_set_capture_enabled", enabled=True)` reaches `Scheduler.tl_set_capture_enabled(...)` via the dispatcher's `getattr(self, method)(**parameters)`.
3. **Capture read-back** — dedicated ZMQ `ipc://` channel. Driver binds PULL before launching Engine; the hook closure lazy-connects PUSH the first time it fires. Per fire, the hook does `socket.send_pyobj({"name": canonical_name, "tensor": cpu_clone})`. Per generate, the driver drains the PULL.

## Locked invariants

Set by `boot_sglang` and not caller-overridable — they're load-bearing for capture correctness:
- `tp_size=1`, `dp_size=1` — multi-device parallelism would split the model across worker subprocesses our single Scheduler patch doesn't address.
- `skip_tokenizer_init=True` — we own tokenization on the bridge side for `to_tokens` consistency.
- `chunked_prefill_size=-1` — chunked prefill splits long prompts into multiple forwards; each fires every hook, so the PULL socket would see N partial messages per hook and the collector would keep just one (silent truncation). Disabled.
- `disable_cuda_graph=True` (default; overridable) — `register_forward_hooks` runs *after* full CUDA graph capture; leaving full graphs on silently bypasses captures during replay.

## Acceptance gates

1. `boot_sglang` returns a `RemoteBridge` end-to-end (worker plugin patched the Scheduler; the ZMQ channel bound; the Engine's spawn handshake completed).
2. Greedy next-token argmax matches `boot_transformers` exactly under fp16+eager attention.
3. Per-hook L2 distance vs `boot_transformers` < `5e-3` (relative).
4. Affine intervention path: `suppress` / `scale` / `add` / `set` each shifts the next-token argmax and reverts on the next clean forward.
5. Capture completeness: one PULL message per supported hook per forward (no missing, no extra). Verifies `chunked_prefill_size=-1` actually took effect and the worker→driver channel didn't drop messages.
6. `get_param` round-trips a real worker tensor (`model.norm.weight`), confirming the side-channel weight read works.
7. `bridge.close()` releases GPU memory + shuts down SGLang's scheduler subprocess + unlinks the ipc socket file.

## Setup

Install SGLang 0.5.12.post1 (the version `assert_sglang_supported()` enforces — `forward_hooks`, the plugin group, and the generic-getattr RPC dispatcher all landed at .12). `pyzmq` is needed for the side channel.

In [ ]:
# Install sglang and TransformerLens @ feature/SGLang-driver. ~3-5 minutes.
# Branch must match this notebook: feature/SGLang-driver has the ZMQ + ServerArgs.forward_hooks design.
# sglang pinned to 0.5.12.post1 — the version the plugin entry-point + forward_hooks shape were validated against.
%pip install -q "sglang==0.5.12.post1" "pyzmq>=25"
%pip install -q git+https://github.com/TransformerLensOrg/TransformerLens.git@feature/SGLang-driver

In [ ]:
import gc
import os
import sys

import torch

# HF_TOKEN comes from Colab secrets. Falls back to env var for non-Colab runs.
try:
    from google.colab import userdata
    os.environ.setdefault("HF_TOKEN", userdata.get("HF_TOKEN"))
except (ImportError, Exception):
    pass
assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN in Colab Secrets (gear icon, left sidebar)."

# Read installed sglang version from package metadata, NOT `import sglang` —
# importing sglang would trigger CUDA C extension loading.
from importlib.metadata import PackageNotFoundError
from importlib.metadata import version as _pkg_version

_PINNED_SGLANG = "0.5.12.post1"
try:
    _sglang_ver = _pkg_version("sglang")
    print(f"sglang version: {_sglang_ver}")
    if _sglang_ver != _PINNED_SGLANG:
        print(
            f"⚠ expected sglang=={_PINNED_SGLANG} (the version this notebook is validated against); "
            f"got {_sglang_ver}. Internal APIs (Scheduler dispatcher, ServerArgs.forward_hooks shape, "
            "tokenizer_manager.model_config.hf_config path) may have drifted. Re-pin and restart."
        )
except PackageNotFoundError:
    print("⚠ sglang is not installed — run the install cell above.")

# Confirm our plugin's entry-point is discoverable; otherwise load_plugins() in
# the worker subprocess won't patch the Scheduler and every collective_rpc call
# will silently no-op (the worker just sees no tl_* methods).
from importlib.metadata import entry_points
_entries = list(entry_points(group="sglang.srt.plugins"))
print(f"sglang.srt.plugins entry points: {[e.name for e in _entries]}")
assert any(e.name == "transformer_lens" for e in _entries), (
    "transformer_lens not registered under sglang.srt.plugins — "
    "reinstall via `pip install -e .` so setuptools picks up the entry-point."
)

MODEL = "meta-llama/Llama-3.2-1B"
PROMPT = "The quick brown fox jumps over the"
DTYPE = torch.float16
torch.manual_seed(0)

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only — abort'}")
assert torch.cuda.is_available(), "GPU runtime required."

## Step 1 — HF reference

Capture activations from the HF transformers backend (the `boot_transformers` driver) as ground truth for Steps 4 & 5. Same dtype + eager attention so the only variable is the SGLang execution path.

In [ ]:
from transformer_lens.model_bridge.transformer_bridge import TransformerBridge

hf_bridge = TransformerBridge.boot_transformers(MODEL, dtype=DTYPE, device="cuda")
tokens = hf_bridge.to_tokens(PROMPT)
print(f"Tokens: shape={tuple(tokens.shape)}, first={tokens[0, :5].tolist()}")

hf_logits, hf_cache = hf_bridge.run_with_cache(tokens)
argmax_hf = int(hf_logits[0, -1].argmax())
print(f"HF next-token argmax: {argmax_hf} → {hf_bridge.tokenizer.decode([argmax_hf])!r}")

hf_acts = {k: v.float().cpu() for k, v in hf_cache.items()}
hf_norm_weight = hf_bridge.original_model.model.norm.weight.float().cpu().clone()
del hf_bridge, hf_logits, hf_cache
gc.collect()
torch.cuda.empty_cache()

## Step 2 — Boot SGLang bridge

`boot_sglang` builds the `ServerArgs.forward_hooks` JSON spec list from the overlay's `capture_specs`, binds the driver-side ZMQ PULL on a fresh `ipc://` channel, then constructs the Engine. SGLang's scheduler subprocess:
1. Re-runs `load_plugins()` (spawn replays interpreter), which fires our entry-point's `register()` → patches the fresh Scheduler class with `tl_*` methods.
2. After `load_model()`, calls `register_forward_hooks(self.model, server_args.forward_hooks)`. SGLang imports our factory by dotted path, calls it per spec, registers the returned hook on each fnmatch-matched module.

First boot takes ~30–60s (weights load, KV pool sizing, piecewise CUDA graph capture).

In [ ]:
from transformer_lens.model_bridge.remote_bridge import RemoteBridge

bridge = RemoteBridge.boot_sglang(
    MODEL,
    dtype=DTYPE,
    # Smaller KV pool than the model's native 128k context — fits on a single L4/A100.
    max_total_tokens=2048,
)
print(f"Bridge: {type(bridge).__name__}")
print(f"Fireable hooks: {len(bridge._driver.supported_hook_points)}")
print(f"PULL channel: {bridge._driver._puller.channel}")
print(f"Non-fireable (kernel-fused): {sorted(h for h in bridge._driver.non_fireable_hook_points if h.startswith('blocks.0'))}")

## Step 3 — Capture pipeline

`run_with_cache` → driver pushes `tl_set_interventions({})` then `tl_set_capture_enabled(True)` → `engine.generate(input_ids=[ids], sampling_params={...}, return_logprob=True, top_logprobs_num=d_vocab)` → driver flips capture off → drains the PULL socket for one message per supported hook → assembles `(1, seq, d_model)`-shaped cache.

In [ ]:
tokens = bridge.to_tokens(PROMPT)
logits, cache = bridge.run_with_cache(tokens)

print(f"Logits: shape={tuple(logits.shape)}, dtype={logits.dtype}")
print(f"Cache hooks: {len(cache)}")
for hk in ["blocks.0.hook_out", "blocks.0.attn.hook_out", "blocks.0.mlp.hook_out"]:
    print(f"  {hk}: shape={tuple(cache[hk].shape)} dtype={cache[hk].dtype}")

## Step 4 — Greedy parity (acceptance gate)

Next-token argmax must match `boot_transformers`. SGLang's sampler bypasses `lm_head.__call__`; the driver synthesizes logits from `meta_info.output_top_logprobs[-1]` (one (logprob, token_id, decoded) tuple per top-k position at the generated step) — argmax-correct, absolute scale off.

In [ ]:
argmax_sglang = int(logits[0, -1].argmax())
parity = argmax_sglang == argmax_hf
print(f"SGLang next-token argmax: {argmax_sglang} → {bridge.tokenizer.decode([argmax_sglang])!r}")
print(f"HF next-token argmax:     {argmax_hf}")
print(f"Greedy parity: {'✓' if parity else '✗'}")
assert parity, "Acceptance gate 2 failed: argmax mismatch vs boot_transformers"

## Step 5 — Per-hook L2 (acceptance gate)

Relative L2 distance per fireable hook must stay under `5e-3` vs HF activations. Larger drifts point at fused-kernel ordering or a wrong overlay path. `ln_final.hook_normalized` is *expected* to mismatch on its own (SGLang exposes post-weight `x*rsqrt(var+eps)*weight`; HF/HT exposes pre-weight `x*rsqrt(var+eps)`) — we explicitly convert it using `get_param`.

In [ ]:
TARGET_REL_L2 = 5e-3
worst_hook, worst_l2 = None, 0.0
for hk in sorted(bridge._driver.supported_hook_points):
    if hk not in cache or hk not in hf_acts:
        continue
    sglang_t = cache[hk].float().cpu()
    if hk == "ln_final.hook_normalized":
        # SGLang gives post-weight; HF gives pre-weight. Divide by the norm weight
        # to recover the HT-equivalent value. Exercises the get_param side channel.
        weight = bridge._driver.get_param("model.norm.weight")
        assert weight is not None, "get_param returned None — tl_get_param round-trip failed"
        sglang_t = sglang_t / weight.float().cpu()
    hf_t = hf_acts[hk]
    rel_l2 = ((sglang_t - hf_t).norm() / (hf_t.norm() + 1e-9)).item()
    flag = "" if rel_l2 < TARGET_REL_L2 else " ← over"
    print(f"  {hk:40s} rel_L2 = {rel_l2:.2e}{flag}")
    if rel_l2 > worst_l2:
        worst_l2, worst_hook = rel_l2, hk
print(f"\nWorst: {worst_hook} = {worst_l2:.2e} (target < {TARGET_REL_L2:.0e})")
assert worst_l2 < TARGET_REL_L2, f"Acceptance gate 3 failed: {worst_hook} = {worst_l2:.2e}"

## Step 6 — Intervention smoke (load-bearing)

Each of `suppress` / `scale` / `add` / `set` must shift next-token argmax and revert on a clean forward (interventions aren't sticky — the driver pushes `tl_set_interventions({})` at the start of every forward to clear stale state). This validates the affine path through the worker hook closure: it reads `_shared_state["interventions"][canonical_name]` on every fire and applies the spec inline before the PUSH send + the downstream return.

Driver-side intervention validation happens in `_validate_interventions` (callables rejected, unsupported ops rejected, unknown hooks rejected). The spec dict travels through `collective_rpc` (kwarg names preserved via `RpcReqInput.parameters`) and arrives at our `Scheduler.tl_set_interventions(specs)`, which writes to the module-global `_shared_state` dict in `hooks.py`.

In [ ]:
# (a) suppress: zero the embedding output.
supp_logits = bridge.forward(tokens, intervene={"embed.hook_out": {"op": "suppress"}})
argmax_supp = int(supp_logits[0, -1].argmax())
print(f"suppress(embed.hook_out): argmax = {argmax_supp} (clean = {argmax_sglang}, shifted = {argmax_supp != argmax_sglang})")

# (b) scale: double the residual stream at layer 0.
scale_logits = bridge.forward(tokens, intervene={"blocks.0.hook_out": {"op": "scale", "factor": 2.0}})
argmax_scale = int(scale_logits[0, -1].argmax())
print(f"scale(blocks.0.hook_out, 2.0): argmax = {argmax_scale} (shifted = {argmax_scale != argmax_sglang})")

# (c) add: shift all residual dims by a constant.
add_logits = bridge.forward(tokens, intervene={"blocks.0.hook_out": {"op": "add", "value": 0.5}})
argmax_add = int(add_logits[0, -1].argmax())
print(f"add(blocks.0.hook_out, 0.5): argmax = {argmax_add} (shifted = {argmax_add != argmax_sglang})")

# (d) set: replace residual with a constant vector.
set_logits = bridge.forward(tokens, intervene={"blocks.0.hook_out": {"op": "set", "value": 0.0}})
argmax_set = int(set_logits[0, -1].argmax())
print(f"set(blocks.0.hook_out, 0): argmax = {argmax_set} (shifted = {argmax_set != argmax_sglang})")

# Clean forward — interventions must NOT persist (driver clears via tl_set_interventions({})).
clean_logits = bridge.forward(tokens)
argmax_clean = int(clean_logits[0, -1].argmax())
print(f"\nclean revert: argmax = {argmax_clean} (matches Step 4 = {argmax_clean == argmax_sglang})")
assert argmax_clean == argmax_sglang, "Acceptance gate 4 failed: intervention leaked into next forward"

## Step 7 — Capture completeness (acceptance gate)

Verifies (a) `chunked_prefill_size=-1` actually disabled chunked prefill — otherwise we'd see N partial messages per hook for an N-chunk prompt, and `_collect_captures`' first-wins would silently truncate. And (b) the worker→driver ZMQ channel didn't drop messages.

Method: count drained messages per hook for one forward. Expected: exactly one per supported hook.

In [ ]:
# Do the forward, then inspect the puller's drain count directly. We monkey-patch
# the puller's drain method temporarily to capture every raw message.
raw_msgs = []
orig_drain = bridge._driver._puller.drain
def _spy_drain(timeout_ms=1000):
    batch = orig_drain(timeout_ms=timeout_ms)
    raw_msgs.extend(batch)
    return batch
bridge._driver._puller.drain = _spy_drain
try:
    _ = bridge.run_with_cache(tokens)
finally:
    bridge._driver._puller.drain = orig_drain

from collections import Counter
counts = Counter(m.get("name") for m in raw_msgs if m.get("name"))
n_supported = len(bridge._driver.supported_hook_points)
print(f"Hooks fired per name (expected exactly 1 each):")
for name in sorted(counts):
    flag = "" if counts[name] == 1 else f"  ← {counts[name]} (chunked prefill leaked?)"
    print(f"  {name:40s} {counts[name]}{flag}")
assert all(c == 1 for c in counts.values()), "Acceptance gate 5 failed: ≥1 hook fired more than once (chunked prefill?)"
assert len(counts) == n_supported, f"Acceptance gate 5 failed: {len(counts)} hooks fired, expected {n_supported}"
print("✓ One message per supported hook, no truncation.")

## Step 8 — `get_param` round-trip (acceptance gate)

Validates the side-channel weight read. The driver calls `engine.collective_rpc("tl_get_param", dotted_name=..., channel=...)` → Scheduler subprocess walks `self.tp_workers[0].model_runner.model.model.norm.weight` → pushes `{"_param": dotted_name, "tensor": cpu_clone}` over the same PUSH socket the hooks use → driver drains PULL and filters for the matching `_param` message.

Already exercised by Step 5's `ln_final` conversion; here we just compare the round-tripped weight against the local HF reference.

In [ ]:
sglang_weight = bridge._driver.get_param("model.norm.weight")
assert sglang_weight is not None, "get_param returned None — tl_get_param dispatch failed"
print(f"Worker weight shape: {tuple(sglang_weight.shape)}, dtype: {sglang_weight.dtype}")
rel_l2 = ((sglang_weight.float() - hf_norm_weight).norm() / (hf_norm_weight.norm() + 1e-9)).item()
print(f"Weight rel_L2 vs HF: {rel_l2:.2e}")
assert rel_l2 < 1e-4, f"Acceptance gate 6 failed: weight round-trip mismatch ({rel_l2:.2e})"

## Step 9 — Lifetime

`bridge.close()` runs `tl_clear_state` (resets the worker's intervention state + disables capture), calls `engine.shutdown()` (which kills the scheduler subprocess + releases GPU memory), then closes the PULL socket and unlinks the `ipc://` socket file (otherwise `/tmp/tl_sglang_*.sock` corpses accumulate).

In [ ]:
channel_path = bridge._driver._puller.channel[len("ipc://"):]
print(f"Socket file before close: exists = {os.path.exists(channel_path)}")
before = torch.cuda.memory_allocated() / 1e9
bridge.close()
gc.collect()
torch.cuda.empty_cache()
after = torch.cuda.memory_allocated() / 1e9
print(f"GPU memory: before close = {before:.2f} GB, after close = {after:.2f} GB")
print(f"Released: {(before - after):.2f} GB")
print(f"Socket file after close:  exists = {os.path.exists(channel_path)} (must be False)")
assert not os.path.exists(channel_path), "Acceptance gate 7 partial: ipc socket file not unlinked"

## Summary

If all seven gates passed, the SGLang driver is correctly:
- Crossing the spawn boundary (worker plugin patched Scheduler, factory ran in the worker process)
- Capturing activations through `register_forward_hooks` (no monkey-patching needed)
- Routing tensors back over a dedicated ZMQ side channel (no overloading of SGLang's ack-only `RpcReqOutput`)
- Applying interventions via the worker-side affine path
- Reading worker weights via the same side channel for HT-equivalent conversion
- Releasing GPU memory + cleaning up `ipc://` socket files on close

**Known v1 scope limits:**
- Single-prompt only (`batch_size=1`, `max_new_tokens=1`). Batched continuous-batching capture needs per-request demarcation on the PULL socket that SGLang's forward context doesn't currently expose to vanilla `register_forward_hook` callables.
- Prompt length bounded by single-forward memory (chunked prefill disabled to keep one-forward-one-message). Tune `mem_fraction_static` for larger prompts.
- TP / DP locked to 1. Multi-device would require multi-Scheduler patching + per-rank channel routing.

Next: the Inspect provider notebook (`tl_bridge_sglang/<model>`) — uses the same boot path under the hood, exposes the same captures through `inspect_ai`'s eval pipeline.